# Claude Code permanente no vlab

O `claude` morre quando o Terminal do Jupyter fecha. Aqui ele roda **dentro do tmux**, que sobrevive a fechar o Terminal ou este notebook, num **laço que o reinicia sozinho** (`claude --continue`, que retoma a conversa desta pasta).

O binário, o login e a configuração ficam em `/home/jovyan/privado/.claude_persist`, que é persistente.

**Limite:** nada sobrevive ao **desligamento do servidor** do Jupyter (*Stop My Server* ou desligamento por ociosidade). Depois de um desligamento ou de uma queda do contêiner, **rode só a célula 0 (Levantar tudo)**. Ela confere e conserta cada peça e diz o que falta. O login é mantido.

| Célula | Quando |
| :--- | :--- |
| **0. Levantar tudo** | **depois de qualquer queda: faz 1, 2, 4 e 4b sozinha e confere** |
| 1. Restaurar | sempre primeiro |
| 2. Instalar / reparar | primeira vez e depois de reiniciar o servidor |
| 3. Login (no Terminal) | só na primeira vez |
| 4. Iniciar | depois de 2; se já estiver rodando, não duplica |
| 4b. Sentinela | depois da 4: manda o estado ao Drive a cada 10 min |
| 5. Estado | a qualquer momento |
| 6. Mandar instrução | sem abrir o Terminal |
| 7. Tela | ler a resposta |
| 7b. Tela ao vivo | acompanhar enquanto ele trabalha (só olha) |
| 7c. Histórico | o que ele fez, com os comandos exatos (só lê) |
| 8. Parar | encerra o Claude e o laço |

**Não use *Run All* para tudo:** as células 6 (Mandar) e 8 (Parar) estão protegidas; é preciso tirar o `#` para elas agirem.

## 0. Levantar tudo (automático)
Roda em ordem e diz **OK** ou **FALHA** em cada passo:
1. Contêiner: se o host mudou (servidor recriado), avisa e lista os modelos que ainda rodam.
2. Ambiente e instalação: `restaurar.sh`; se o `claude` ou o `tmux` sumiram, roda `instalar`; senão só `reparar`.
3. Sessão `claude`: cria, se não existir.
4. Tela do Claude: **espera até ele abrir** (até `ESPERA_CLAUDE_S`) e reconhece a tela: pronto, pedindo login, perguntando se confia na pasta, ou com erro de configuração. Nesse último caso, conserta sozinha.
5. Reinícios: avisa se o Claude está caindo em laço.
6. Sentinela: liga, se estiver desligada, e **confere que o estado chegou ao Drive**.

Pode rodar quantas vezes quiser: o que já está certo não é refeito. Se aparecer **login** ou **confiança**, a célula diz o que fazer no Terminal.

In [ ]:
# 0. Levantar tudo: confere cada peca, conserta o que der e diz o que falta (pode rodar quantas vezes quiser)
import os, re, shutil, subprocess, time, json, socket, datetime as dt

AQUI = '/home/jovyan/privado/Himalaia_2026/toro_cm1_lhende'
PRIV = '/home/jovyan/privado'
AO_VIVO = 'gdrive:vlab_cache/Himalaia_2026/saida/toro_cm1_lhende/ao_vivo'
ESPERA_CLAUDE_S = 150     # quanto esperar a tela do Claude ficar pronta (ele demora a abrir num conteiner novo)
ESPERA_DRIVE_S = 180      # quanto esperar a sentinela gravar o estado no Drive
CONFIAR_PASTA = False     # True: se o Claude perguntar se confia nesta pasta, responde Sim sozinho

TMUX = next((p for p in [PRIV + '/conda_envs/ferramentas/bin/tmux', shutil.which('tmux') or ''] if p and os.path.exists(p)), None)
res = []                  # (passo, ok?, detalhe)

def sh(cmd, timeout=900):
    """roda no bash de login, na pasta do projeto e com o ambiente restaurado"""
    try:
        r = subprocess.run(['bash', '-lc', 'cd %s && source restaurar.sh >/dev/null 2>&1; %s' % (AQUI, cmd)],
                           capture_output=True, text=True, timeout=timeout)
        return r.returncode, (r.stdout + r.stderr).strip()
    except subprocess.TimeoutExpired:
        return 124, 'tempo esgotado (%d s)' % timeout

def tmux(*a):
    if not TMUX:
        return 1, 'sem tmux'
    r = subprocess.run([TMUX, *a], capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)

def sessao(nome):
    return tmux('has-session', '-t', nome)[0] == 0

def tela(n=30):
    rc, out = tmux('capture-pane', '-p', '-t', 'claude')
    ls = [l for l in out.splitlines() if l.strip()] if rc == 0 else []
    return '\n'.join(ls[-n:])

def classifica(t):
    fim = '\n'.join(t.splitlines()[-15:])
    if not t.strip():                                        return 'vazia'
    if 'Configuration error' in fim:                         return 'erro_config'
    if re.search(r'Select login method|Paste code here|Browser didn.t open|sign in|log ?in to', fim, re.I): return 'login'
    if re.search(r'trust (this|the files in this) folder|Do you trust', fim, re.I): return 'confianca'
    if re.search(r'❯|\? for shortcuts|bypass permissions|auto mode|esc to interrupt', fim): return 'pronto'
    return 'outro'

def passo(nome, ok, det=''):
    res.append((nome, ok, det)); print(('OK   ' if ok else 'FALHA') + '  ' + nome + ('  -- ' + det if det else ''), flush=True)

print(time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()), '| host', socket.gethostname(), '\n')

# 1. host novo? (o conteiner foi recriado: tudo o que rodava morreu)
reg = os.path.join(AQUI, 'logs', 'ultimo_host.txt'); os.makedirs(os.path.dirname(reg), exist_ok=True)
antes = open(reg).read().strip() if os.path.exists(reg) else ''
agora = socket.gethostname()
if antes and antes != agora:
    passo('conteiner', False, 'host NOVO (%s -> %s): o servidor foi recriado; rodadas que estavam rodando morreram' % (antes, agora))
else:
    passo('conteiner', True, 'mesmo host (%s)' % agora if antes else 'primeiro registro deste host')
open(reg, 'w').write(agora)
rc, procs = sh(r"ps -eo args | grep -E '[w]rf_[A-Z]\.exe|[r]eal_[A-Z]\.exe|[c]m1\.exe' | grep -oE '(wrf|real)_[A-Z]\.exe|cm1\.exe' | sort | uniq -c")
print('       modelos rodando agora (processos):', ' '.join(procs.split()) if procs else 'nenhum')

# 2. ambiente e instalacao (links do login e do binario na area persistente; tmux)
rc, out = sh('bash restaurar.sh')
passo('restaurar.sh', rc == 0, '' if rc == 0 else out[-300:])
rc_v, ver = sh('export PATH=$HOME/.local/bin:$PATH; claude --version 2>/dev/null | head -1')
if not TMUX or rc_v != 0 or not ver:
    rc, out = sh('bash claude_permanente.sh instalar', timeout=1200)
    TMUX = next((p for p in [PRIV + '/conda_envs/ferramentas/bin/tmux', shutil.which('tmux') or ''] if p and os.path.exists(p)), None)
    rc_v, ver = sh('export PATH=$HOME/.local/bin:$PATH; claude --version 2>/dev/null | head -1')
    passo('instalar', rc == 0 and bool(ver), (ver or out[-300:]))
else:
    sh('bash claude_permanente.sh reparar')          # so refaz links e confere o .claude.json (rapido)
    passo('instalacao', True, 'tmux %s; claude %s' % (os.path.basename(TMUX), ver))

# 3. sessao do Claude
if not sessao('claude'):
    rc, out = sh('bash claude_permanente.sh iniciar')
    passo('iniciar sessao', sessao('claude'), out.splitlines()[-1] if out else '')
else:
    passo('sessao claude', True, 'ja existia')

# 4. espera a tela ficar pronta e trata as telas conhecidas
estado, t0, consertou = 'vazia', time.time(), set()
while time.time() - t0 < ESPERA_CLAUDE_S:
    estado = classifica(tela())
    if estado == 'erro_config' and 'cfg' not in consertou:
        sh('bash claude_permanente.sh reparar'); consertou.add('cfg'); time.sleep(15); continue
    if estado == 'confianca' and CONFIAR_PASTA and 'trust' not in consertou:
        tmux('send-keys', '-t', 'claude', 'Enter'); consertou.add('trust'); time.sleep(5); continue
    if estado in ('pronto', 'login') or (estado == 'confianca' and not CONFIAR_PASTA):
        break
    time.sleep(5)
dica = {'pronto': 'pronto para receber ordens',
        'login': 'PEDE LOGIN: no Terminal rode  tmux attach -t claude  , faca o login e saia com Ctrl+B, D',
        'confianca': 'pergunta se confia na pasta: no Terminal  tmux attach -t claude  e escolha Yes (ou rode de novo com CONFIAR_PASTA = True)',
        'erro_config': 'erro de configuracao persiste mesmo apos reparar: veja a tela abaixo',
        'vazia': 'a tela continua vazia apos %d s: o Claude nao abriu (veja o log abaixo)' % ESPERA_CLAUDE_S,
        'outro': 'tela nao reconhecida (veja abaixo); pode estar trabalhando'}[estado]
passo('tela do Claude', estado in ('pronto', 'outro'), dica)

# 5. laco de quedas? (muitos reinicios recentes)
lg = os.path.join(AQUI, 'logs', 'claude_permanente.log')
if os.path.exists(lg):
    agora_utc = dt.datetime.utcnow(); n = 0
    for l in open(lg, errors='ignore').read().splitlines()[-40:]:
        m = re.match(r'\[(\d{4}-\d\d-\d\dT\d\d:\d\d:\d\d)Z\] iniciando', l)
        if m and (agora_utc - dt.datetime.strptime(m.group(1), '%Y-%m-%dT%H:%M:%S')).total_seconds() < 600:
            n += 1
    passo('reinicios nos ultimos 10 min', n <= 2, '%d%s' % (n, ' -- o Claude esta caindo em laco; veja o log' if n > 2 else ''))

# 6. sentinela e Drive
if not sessao('sentinela'):
    rc, out = sh('bash claude_permanente.sh sentinela')
    passo('iniciar sentinela', sessao('sentinela'), out.splitlines()[-1] if out else '')
else:
    passo('sentinela', True, 'ja rodando')
t0, idade = time.time(), None
while time.time() - t0 < ESPERA_DRIVE_S:
    rc, out = sh('rclone lsjson %s/estado_vlab.txt 2>/dev/null' % AO_VIVO, timeout=120)
    try:
        mt = json.loads(out)[0]['ModTime'][:19]
        idade = (dt.datetime.utcnow() - dt.datetime.strptime(mt, '%Y-%m-%dT%H:%M:%S')).total_seconds()
    except Exception:
        idade = None
    if idade is not None and idade < 150:
        break
    time.sleep(20)
passo('estado no Drive', idade is not None and idade < 150,
      ('atualizado ha %d s' % idade) if idade is not None else 'nao consegui ler o Drive (rclone)')

# resumo
print('\n' + ('TUDO OK' if all(ok for nome, ok, d in res if nome != 'conteiner') else 'ATENCAO: ha itens com FALHA acima'))
print('\n---- ultimas linhas da tela do Claude ----\n' + (tela(20) or '(vazia)'))
if os.path.exists(lg):
    print('\n---- log do laco ----\n' + '\n'.join(open(lg, errors='ignore').read().splitlines()[-4:]))

## 1. Restaurar o ambiente

In [ ]:
%%bash -l
cd /home/jovyan/privado/Himalaia_2026/toro_cm1_lhende
bash restaurar.sh


## 2. Instalar / reparar
Instala o tmux (conda, em `conda_envs/ferramentas`) e o Claude Code, se faltarem, e refaz os links para a área persistente. Se o Claude Code já está instalado, só confere.

In [ ]:
%%bash -l
cd /home/jovyan/privado/Himalaia_2026/toro_cm1_lhende
bash claude_permanente.sh instalar


## 3. Login (só na primeira vez, no **Terminal**)
Rode a célula 4 e depois, num Terminal do Jupyter (*File → New → Terminal*):
```
tmux attach -t claude
```
Faça o login (o `claude` mostra um link; abra no navegador do seu computador e cole o código). Na pergunta de confiança, escolha **Yes, I trust this folder**. Para sair **sem encerrar** o Claude: **Ctrl+B** e depois **D**.

O login fica em `/home/jovyan/privado/.claude_persist` e vale para as próximas vezes.

## 4. Iniciar
Cria a sessão tmux `claude`, com reinício automático. Se ela já existe, não duplica.

In [ ]:
%%bash -l
cd /home/jovyan/privado/Himalaia_2026/toro_cm1_lhende
bash claude_permanente.sh iniciar


## 4b. Sentinela (recomendado)
A cada 10 min grava o estado do Claude (sessão e últimas linhas da tela) e da fila do CM1 em `gdrive:vlab_cache/Himalaia_2026/saida/toro_cm1_lhende/ao_vivo/estado_vlab.txt`. No Windows: `G:\Meu Drive\vlab_cache\Himalaia_2026\saida\toro_cm1_lhende\ao_vivo\`. Assim a sessão do Claude no seu computador verifica o vlab sozinha, sem você copiar e colar. Também roda no tmux e sobrevive a fechar o notebook.

In [ ]:
%%bash -l
cd /home/jovyan/privado/Himalaia_2026/toro_cm1_lhende
bash claude_permanente.sh sentinela


## 5. Estado
Mostra se a sessão está viva, o histórico de reinícios e as últimas linhas da tela do Claude.

In [ ]:
%%bash -l
cd /home/jovyan/privado/Himalaia_2026/toro_cm1_lhende
bash claude_permanente.sh estado


## 6. Mandar uma instrução (sem abrir o Terminal)
Troque o texto de `MSG`. Ele é digitado no Claude, seguido de Enter. Para ver a resposta, use a célula 7 depois de alguns segundos ou minutos.

Exemplos:
* `Leia a ordem com rclone cat gdrive:vlab_cache/Himalaia_2026/entrada/ORDEM_reproducao_rad06.md e execute-a. Eu autorizo.`
* `/remote-control` — liga o Remote Control, para a sessão poder ser alcançada de fora, se o seu plano e o contêiner permitirem.

In [ ]:
%%bash -l
cd /home/jovyan/privado/Himalaia_2026/toro_cm1_lhende
MSG='Rode bash rodar_direto.sh estado e me diga o resultado.'
# PROTEGIDA contra 'Run All': tire o # da linha abaixo para enviar
# bash claude_permanente.sh mandar "$MSG"
echo 'celula protegida: nada foi enviado (tire o # para mandar)'


## 7. Tela
Últimas linhas da tela do Claude (troque 80 por outro número, se quiser).

In [ ]:
%%bash -l
cd /home/jovyan/privado/Himalaia_2026/toro_cm1_lhende
bash claude_permanente.sh tela 80


## 7b. Tela ao vivo
Mostra a tela do Claude atualizando a cada 5 s, por `MINUTOS` minutos (padrão 10). **Só olha**: nada é digitado no Claude.
O botão Interromper (■) para a exibição; o Claude continua rodando. Num *Run All*, esta célula segura a execução pelos `MINUTOS`.

In [ ]:
import os, shutil, subprocess, time
from IPython.display import clear_output
MINUTOS = 10
TMUX = '/home/jovyan/privado/conda_envs/ferramentas/bin/tmux'
TMUX = TMUX if os.path.exists(TMUX) else (shutil.which('tmux') or TMUX)
try:
    for i in range(MINUTOS * 12):
        s = subprocess.run([TMUX, 'capture-pane', '-p', '-t', 'claude', '-S', '-45'], capture_output=True, text=True)
        clear_output(wait=True)
        print(time.strftime('%H:%M:%S'), '| ao vivo, atualiza a cada 5 s (Interromper para parar)\n')
        print(s.stdout or s.stderr or 'sessao claude nao encontrada (rode as celulas 1, 2 e 4)')
        time.sleep(5)
except KeyboardInterrupt:
    print('exibicao parada (o Claude continua rodando)')


## 7c. O que o Claude fez (histórico com os comandos)
Lê o histórico da conversa que o próprio Claude Code grava e mostra os últimos `N` eventos:
* **VOCE**: as suas mensagens;
* **CLAUDE**: as respostas dele;
* **COMANDO**: cada comando que ele rodou, exatamente como rodou;
* **RESULTADO**: o fim da saída de cada comando.

Os horários estão em UTC. **Só lê**: não muda nada.

In [ ]:
import json, glob, os
N = 40                      # quantos eventos mostrar (os ultimos)
P = '/home/jovyan/privado/.claude_persist/.claude/projects'
arqs = glob.glob(P + '/*/*.jsonl')
if not arqs:
    print('nenhum historico em', P)
else:
    f = max(arqs, key=os.path.getmtime)
    ev = []
    for l in open(f, encoding='utf-8'):
        try: d = json.loads(l)
        except Exception: continue
        c = (d.get('message') or {}).get('content'); t = d.get('timestamp', '')[11:19] + ' UTC'
        if isinstance(c, str):
            ev.append((t, 'VOCE' if d.get('type') == 'user' else 'CLAUDE', c)); continue
        for b in c or []:
            if not isinstance(b, dict): continue
            k = b.get('type')
            if k == 'text': ev.append((t, 'CLAUDE', b.get('text', '')))
            elif k == 'tool_use':
                i = b.get('input') or {}
                ev.append((t, 'COMANDO', i.get('command') or json.dumps(i, ensure_ascii=False)[:400]))
            elif k == 'tool_result':
                r = b.get('content'); r = r if isinstance(r, str) else ' '.join(x.get('text', '') for x in (r or []) if isinstance(x, dict))
                ev.append((t, 'RESULTADO', r[-700:]))
    print('historico:', f, '| eventos:', len(ev), '\n')
    for t, k, x in ev[-N:]:
        print(f'[{t}] {k}:\n  ' + x.strip()[:1500].replace('\n', '\n  ') + '\n')


## 8. Parar
Encerra a sessão tmux e o laço de reinício. As rodadas do CM1 lançadas pelo `rodar_direto.sh` **continuam**; elas não dependem do Claude.

Para parar só a sentinela: `bash claude_permanente.sh parar_sentinela`.

In [ ]:
%%bash -l
cd /home/jovyan/privado/Himalaia_2026/toro_cm1_lhende
# PROTEGIDA contra 'Run All': tire o # da linha abaixo para encerrar o Claude de proposito
# bash claude_permanente.sh parar
echo 'celula protegida: nada foi encerrado (tire o # para parar)'
